# Import Libraries

Import necessary libraries such as pandas, numpy, matplotlib, seaborn, and any others required for data analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Load Raw Data

Load the raw e-commerce data from CSV files into pandas DataFrames.

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/processed/cleaned_data.csv')

print("Data loaded successfully")
print("Shape:", df.shape)
display(df.head())

# Data Cleaning and Preprocessing

Clean the data by handling missing values, duplicates, data types, and outliers. Prepare the data for analysis.

In [ ]:
# Assuming data is cleaned
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivered_date'] = pd.to_datetime(df['delivered_date'], errors='coerce')
df['estimated_delivery_date'] = pd.to_datetime(df['estimated_delivery_date'], errors='coerce')

# Additional preprocessing for funnel analysis
df['delivery_time'] = (df['delivered_date'] - df['order_date']).dt.days
df['on_time_delivery'] = (df['delivered_date'] <= df['estimated_delivery_date']).astype(int)

print("Data preprocessing complete")

# Exploratory Data Analysis (EDA)

Perform EDA to understand data distributions, correlations, and key statistics using plots and summary statistics.

In [ ]:
# Basic EDA for funnel analysis
print("Key statistics for funnel analysis:")
print(df.describe())

# Check unique values in categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"\n{col} unique values:")
    print(df[col].value_counts().head())

# Funnel Analysis

Analyze the customer journey funnel, calculating drop-off rates at each stage.

In [ ]:
# Funnel analysis based on order status
order_status_counts = df['order_status'].value_counts()
print("Order Status Distribution:")
display(order_status_counts)

# Since all are delivered, analyze delivery performance
total_orders = len(df)
on_time_deliveries = df['on_time_delivery'].sum()
on_time_rate = on_time_deliveries / total_orders * 100

print(f"\nTotal Delivered Orders: {total_orders}")
print(f"On-Time Deliveries: {on_time_deliveries}")
print(f"On-Time Delivery Rate: {on_time_rate:.1f}%")

# Funnel visualization: Order -> Delivered On Time
funnel_data = pd.DataFrame({
    'stage': ['Ordered', 'Delivered On Time'],
    'count': [total_orders, on_time_deliveries]
})

plt.figure(figsize=(8, 5))
sns.barplot(x='stage', y='count', data=funnel_data)
plt.title('Delivery Performance Funnel')
plt.show()

# Delivery time analysis
avg_delivery_time = df['delivery_time'].mean()
median_delivery_time = df['delivery_time'].median()

print(f"\nAverage Delivery Time: {avg_delivery_time:.1f} days")
print(f"Median Delivery Time: {median_delivery_time:.1f} days")

plt.figure(figsize=(10, 6))
sns.histplot(df['delivery_time'], bins=30, kde=True)
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time (days)')
plt.show()

# Conversion Rate Calculation

Compute conversion rates for different stages of the funnel and segments.

In [ ]:
# Conversion rates by channel or segment
if 'channel' in df.columns:
    conversion_by_channel = df.groupby('channel').agg({
        'order_id': 'count',
        'customer_id': 'nunique'
    }).rename(columns={'order_id': 'orders', 'customer_id': 'customers'})
    
    conversion_by_channel['conversion_rate'] = conversion_by_channel['orders'] / conversion_by_channel['customers']
    
    print("Conversion rates by channel:")
    display(conversion_by_channel)
    
    # Visualize
    plt.figure(figsize=(8, 5))
    sns.barplot(x=conversion_by_channel.index, y='conversion_rate', data=conversion_by_channel)
    plt.title('Conversion Rate by Channel')
    plt.show()
else:
    print("No channel column found for segmented conversion analysis")

# Cohort Analysis

Group customers into cohorts based on acquisition date and analyze retention and behavior over time.

In [ ]:
# Cohort analysis
# Find first purchase date for each customer
customer_first_purchase = df.groupby('customer_unique_id')['order_date'].min().reset_index()
customer_first_purchase.columns = ['customer_unique_id', 'cohort_date']

# Merge back
df_with_cohort = df.merge(customer_first_purchase, on='customer_unique_id')

# Create cohort index (months since first purchase)
df_with_cohort['cohort_index'] = ((df_with_cohort['order_date'].dt.year - df_with_cohort['cohort_date'].dt.year) * 12 + 
                                  (df_with_cohort['order_date'].dt.month - df_with_cohort['cohort_date'].dt.month))

# Cohort retention
cohort_data = df_with_cohort.groupby(['cohort_date', 'cohort_index'])['customer_unique_id'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='cohort_date', columns='cohort_index', values='customer_unique_id')

# Calculate retention rates
cohort_sizes = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_sizes, axis=0)

print("Cohort Retention Heatmap:")
plt.figure(figsize=(12, 8))
sns.heatmap(retention, annot=True, fmt='.0%', cmap='Blues')
plt.title('Cohort Retention Analysis')
plt.show()

# Customer Segmentation

Segment customers using clustering techniques like K-means based on purchase behavior and demographics.

In [ ]:
# Customer segmentation (similar to EDA notebook)
customer_features = df.groupby('customer_unique_id').agg({
    'total': 'sum',
    'order_id': 'count',
    'price': 'mean'
}).rename(columns={'order_id': 'num_orders', 'total': 'total_revenue', 'price': 'avg_order_value'})

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_features = scaler.fit_transform(customer_features)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_features['segment'] = kmeans.fit_predict(scaled_features)

print("Customer segments summary:")
display(customer_features.groupby('segment').mean())

# Visualize segments
plt.figure(figsize=(10, 6))
sns.scatterplot(x='total_revenue', y='num_orders', hue='segment', data=customer_features, palette='viridis')
plt.title('Customer Segmentation')
plt.show()

# Revenue Segmentation

Analyze revenue by segments, products, or time periods to identify key drivers.

In [ ]:
# Revenue by segment
revenue_by_segment = customer_features.groupby('segment')['total_revenue'].sum()

print("Revenue by customer segment:")
display(revenue_by_segment)

plt.figure(figsize=(8, 5))
revenue_by_segment.plot(kind='bar')
plt.title('Revenue by Customer Segment')
plt.ylabel('Total Revenue')
plt.show()

# Revenue over time
revenue_over_time = df.groupby(df['order_date'].dt.to_period('M'))['total'].sum()

plt.figure(figsize=(12, 6))
revenue_over_time.plot()
plt.title('Monthly Revenue Trend')
plt.ylabel('Revenue')
plt.show()

# Visualization and Dashboard Preparation

Create visualizations for key metrics and prepare data for dashboard tools like Tableau.

In [ ]:
# Additional visualizations
# Top products by revenue
if 'product_id' in df.columns:
    top_products = df.groupby('product_id')['total_revenue'].sum().nlargest(10)
    plt.figure(figsize=(10, 6))
    top_products.plot(kind='bar')
    plt.title('Top 10 Products by Revenue')
    plt.xticks(rotation=45)
    plt.show()

# Save processed data for dashboard
df.to_csv('../data/processed/dashboard_data.csv', index=False)
print("Data saved for dashboard use")

# Generate Business Insights

Summarize findings into actionable insights, such as recommendations for improving conversion or targeting segments.

In [ ]:
# Business Insights
insights = []

# Delivery insights
insights.append(f"On-time delivery rate is {on_time_rate:.1f}%. Focus on improving logistics for better customer satisfaction.")

# Segment insights
high_value_segment = revenue_by_segment.idxmax()
insights.append(f"Segment {high_value_segment} generates the most revenue. Target marketing efforts here.")

# Cohort insights
avg_retention_3m = retention.iloc[:, 3].mean() if retention.shape[1] > 3 else retention.iloc[:, -1].mean()
insights.append(f"Average retention rate is {avg_retention_3m:.1%} after 3 months. Consider strategies to improve retention.")

print("Key Business Insights:")
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

# Save insights
with open('../insights/recommendations.md', 'w') as f:
    f.write('# Business Recommendations\n\n')
    for insight in insights:
        f.write(f'- {insight}\n')

print("Insights saved to recommendations.md")